# Epsilon Fund — MoneyIn Long + BTC Regime Filter

Tests whether the HMM/XGBoost regime classifier adds value when applied as a gate to the **MoneyIn Long** strategy — a directional long-only BTC trend strategy with ATR-trailing stops.

**Strategy:** EMA(14) entry + swing-high ATR trailing stop  
**Filter source:** `topics/regime-classifier/data/predictions/btc_regime_predictions.parquet`

**Variants compared:**

| Variant | Regime gate |
|---|---|
| Baseline | No filter — trade whenever signal fires |
| Bull + Chop | Allow regimes 0 (Bull) and 1 (Recovery/Chop) |
| Bull only | Allow regime 0 (Bull) only — most conservative |

---

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'

sys.path.append(ROOT + r'\infrastructure\data')
sys.path.append(ROOT + r'\infrastructure\backtester')

from engine import backtest

CACHE_DIR = ROOT + r'\live_trading\cache\daily'
PREDS     = ROOT + r'\topics\regime-classifier\data\predictions\btc_regime_predictions.parquet'

COST = 0.001
REGIME_NAMES = ['Bull', 'Recovery/Chop', 'Bear', 'Extreme Bear']

---
## Data

In [ ]:
# BTC daily from cache (already up-to-date)
btc = pd.read_parquet(CACHE_DIR + r'\BTCUSDT_daily.parquet')
btc.index = pd.to_datetime(btc.index).tz_localize(None)
btc = btc.sort_index()
print(f'BTC daily: {btc.index[0].date()} to {btc.index[-1].date()}  ({len(btc)} rows)')

# Regime predictions
regime_df = pd.read_parquet(PREDS)
regime_df.index = pd.to_datetime(regime_df.index).tz_localize(None)
print(f'Regime predictions: {regime_df.index[0].date()} to {regime_df.index[-1].date()}  ({len(regime_df)} rows)')
print()
print('Regime distribution:')
for r, cnt in regime_df['pred_regime'].value_counts().sort_index().items():
    print(f'  Regime {r} ({REGIME_NAMES[int(r)]}): {cnt} days ({cnt/len(regime_df)*100:.1f}%)')

---
## Strategy — MoneyIn Long

Long-only trend strategy on BTC daily:
- **Entry:** Close > EMA(14) and no caution flag (extended swing high)
- **Exit:** Close crosses below ATR-based trailing stop anchored to rolling swing high
- **Stop tightens** when caution flag is active (0.4× ATR) vs normal (1.0× ATR)

No parameters are optimised — this is a fixed-parameter strategy, so no walk-forward is needed.

In [ ]:
df = btc.copy()

# Indicators
df['EMA_14']       = df['Close'].ewm(span=14, adjust=False).mean()
df['Swing_High_7'] = df['High'].rolling(7).max()
df['Swing_High_3'] = df['High'].rolling(5).max()
df['Swing_Low_3']  = df['Low'].rolling(5).min()

# ATR(21)
hi, lo, cl = df['High'], df['Low'], df['Close']
tr = pd.concat([(hi - lo), (hi - cl.shift()).abs(), (lo - cl.shift()).abs()], axis=1).max(axis=1)
df['ATR_21'] = tr.rolling(21).mean()

# Caution flag: swing range > 1.5× ATR or below EMA
df['Caution'] = (
    ((df['Swing_High_7'] - df['Low']) > 1.5 * df['ATR_21']) |
    (df['Close'] < df['EMA_14'])
)

# Entry signal
df['Entry'] = (df['Close'] > df['EMA_14']) & (~df['Caution'])

# Position loop
df['position']   = 0
df['stop_level'] = np.nan

in_pos    = False
stop_loss = 0.0

for i in range(21, len(df)):
    idx      = df.index[i]
    prev_idx = df.index[i - 1]
    curr     = df.loc[idx]
    prev     = df.loc[prev_idx]

    stop_mult     = 0.4 if curr['Caution'] else 1.0
    stop_distance = curr['ATR_21'] * stop_mult * 1.1

    if in_pos:
        if prev['Close'] < stop_loss:
            in_pos = False
            df.loc[idx, 'position'] = 0
            continue
        stop_loss = max(stop_loss, curr['Swing_High_3'] - stop_distance)
        df.loc[idx, ['position', 'stop_level']] = [1, stop_loss]
    else:
        if prev['Entry']:
            stop_loss = curr['Swing_High_3'] - stop_distance
            in_pos    = True
            df.loc[idx, ['position', 'stop_level']] = [1, stop_loss]

df = df.dropna(subset=['EMA_14', 'ATR_21', 'Swing_High_7'])
df['position'] = df['position'].fillna(0).astype(int)

n_long = (df['position'] == 1).sum()
n_flat = (df['position'] == 0).sum()
print(f'Strategy signal: {len(df)} days  |  Long: {n_long} ({n_long/len(df)*100:.1f}%)  Flat: {n_flat} ({n_flat/len(df)*100:.1f}%)')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')

---
## Apply Regime Filter

The regime mask zeros out the `position` on days where the predicted BTC regime is not in the allowed set.  
Days before the regime predictions start (before 2021-11-25) default to **allow** (mask = 1).

In [ ]:
def apply_regime_filter(df, regime_df, allow_regimes):
    mask = regime_df['pred_regime'].isin(allow_regimes).astype(int)
    mask = mask.reindex(df.index).fillna(1)   # default allow before predictions start
    d = df.copy()
    d['position'] = (d['position'] * mask).astype(int)
    return d

df_base      = df.copy()                                                          # no filter
df_bullchop  = apply_regime_filter(df, regime_df, allow_regimes=(0, 1))          # Bull + Chop
df_bull      = apply_regime_filter(df, regime_df, allow_regimes=(0,))            # Bull only

# Coverage stats
overlap = df.index[df.index.isin(regime_df.index)]
reg_on_days = regime_df['pred_regime'].reindex(overlap)

print('Regime filter coverage (days with predictions):')
print(f'  Total strategy days      : {len(df)}')
print(f'  Days with regime data    : {len(overlap)}')
print()
for label, allow in [('Bull+Chop', (0,1)), ('Bull only', (0,))]:
    blocked = (~reg_on_days.isin(allow)).sum()
    pct     = blocked / len(overlap) * 100
    print(f'  {label:<12} blocked {blocked} days ({pct:.1f}%) of signal days')

---
## Backtest All Variants

In [ ]:
variants = [
    ('Baseline (no filter)',     df_base),
    ('Bull + Chop filter',       df_bullchop),
    ('Bull-only filter',         df_bull),
]

results = {}
for label, d in variants:
    r = backtest(d, cost=COST, show_plot=False, save_html=None)
    results[label] = r
    print(f'{label}')
    print(f'  Return {r["total_return"]*100:>8.2f}%  |  Sharpe {r["sharpe_ratio"]:>5.2f}  |  MaxDD {r["max_drawdown"]*100:>8.2f}%  |  PF {r["profit_factor"]:>5.2f}  |  WR {r["win_rate"]*100:>5.1f}%  |  Trades {r["num_trades"]:>4}')
    print()

# Side-by-side table
print('=' * 92)
print(f'  {"Strategy":<28}  {"Return":>9}  {"Sharpe":>7}  {"MaxDD":>8}  {"Calmar":>7}  {"PF":>6}  {"WR":>7}  {"Trades":>7}')
print('  ' + '-' * 88)
for label, r in results.items():
    cal = r['total_return'] / abs(r['max_drawdown']) if r['max_drawdown'] != 0 else 0
    print(f'  {label:<28}  {r["total_return"]*100:>8.1f}%  {r["sharpe_ratio"]:>7.2f}  '
          f'{r["max_drawdown"]*100:>7.1f}%  {cal:>7.2f}  {r["profit_factor"]:>6.2f}  '
          f'{r["win_rate"]*100:>6.1f}%  {r["num_trades"]:>7}')

---
## Equity Curves

In [ ]:
def equity_curve(d, cost):
    pos   = d['position'].shift(1).fillna(0)
    ret   = d['Close'].pct_change()
    strat = pos * ret
    trade = (pos != pos.shift(1)).astype(float) * cost
    return (1 + strat - trade).cumprod().dropna()

# BTC buy-and-hold over the same period
bnh = (1 + df['Close'].pct_change()).cumprod().dropna()

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(3, 1, height_ratios=[4, 2, 1.5], hspace=0.06)

ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax2 = fig.add_subplot(gs[2], sharex=ax0)

colors = {
    'Baseline (no filter)':  '#546e7a',
    'Bull + Chop filter':    '#1565c0',
    'Bull-only filter':      '#2e7d32',
}

# Equity curves
for label, d in variants:
    eq = equity_curve(d, COST)
    ax0.plot(eq.index, eq.values, label=label, color=colors[label], linewidth=1.5)

ax0.plot(bnh.index, bnh.values, label='BTC Buy & Hold', color='#bdbdbd',
         linewidth=1.0, linestyle='--', alpha=0.7)
ax0.axhline(1, color='black', linewidth=0.4, linestyle='--')
ax0.set_ylabel('Cumulative return')
ax0.set_title('MoneyIn Long — Baseline vs Regime Filter variants')
ax0.legend(fontsize=9)
plt.setp(ax0.get_xticklabels(), visible=False)

# Drawdown — best filtered vs baseline
for label, d in [('Baseline (no filter)', df_base), ('Bull-only filter', df_bull)]:
    eq = equity_curve(d, COST)
    dd = (eq / eq.cummax() - 1) * 100
    ax1.fill_between(dd.index, dd.values, 0, alpha=0.35, color=colors[label], label=label)
ax1.set_ylabel('Drawdown %')
ax1.legend(fontsize=8)
plt.setp(ax1.get_xticklabels(), visible=False)

# Regime colour strip
reg_aligned = regime_df['pred_regime'].reindex(df.index)
regime_colors_map = {0: '#1565c0', 1: '#2e7d32', 2: '#f57c00', 3: '#d32f2f'}
bar_colors = [regime_colors_map.get(int(r), '#bdbdbd') if pd.notna(r) else '#bdbdbd'
              for r in reg_aligned]
ax2.bar(df.index, np.ones(len(df)), color=bar_colors, width=1.5, align='edge', linewidth=0)
ax2.set_ylabel('BTC Regime')
ax2.set_ylim(0, 1)
ax2.set_yticks([])

# Legend patches for regime strip
from matplotlib.patches import Patch
regime_legend = [Patch(color=c, label=REGIME_NAMES[i]) for i, c in regime_colors_map.items()]
ax2.legend(handles=regime_legend, fontsize=7, loc='lower left', ncol=4)

plt.tight_layout()
plt.show()

---
## Yearly Breakdown

In [ ]:
def yearly_returns(d, cost):
    pos   = d['position'].shift(1).fillna(0)
    ret   = d['Close'].pct_change()
    strat = pos * ret - (pos != pos.shift(1)).astype(float) * cost
    eq    = (1 + strat).cumprod().dropna()
    yearly = {}
    for yr, grp in eq.groupby(eq.index.year):
        yearly[yr] = grp.iloc[-1] / grp.iloc[0] - 1
    return yearly

years = sorted({yr for d in [df_base, df_bullchop, df_bull] for yr in yearly_returns(d, COST)})

print(f'  {"Year":<6}  {"Baseline":>10}  {"Bull+Chop":>11}  {"Bull-only":>11}  {"BTC B&H":>10}')
print('  ' + '-' * 54)

bnh_yr = yearly_returns(df.assign(position=1), COST)

for yr in years:
    b  = yearly_returns(df_base,     COST).get(yr, float('nan'))
    bc = yearly_returns(df_bullchop, COST).get(yr, float('nan'))
    bo = yearly_returns(df_bull,     COST).get(yr, float('nan'))
    bh = bnh_yr.get(yr, float('nan'))
    print(f'  {yr:<6}  {b*100:>9.1f}%  {bc*100:>10.1f}%  {bo*100:>10.1f}%  {bh*100:>9.1f}%')

---
## Probability Threshold Filter

Instead of a hard regime label gate (`pred_regime == 0`), use the raw **Bull probability** `p_regime_0`.
Only allow a position when the model is confident: `p_regime_0 > threshold`.

| Mode | Logic |
|---|---|
| **Pure** | `p_regime_0 > threshold` only |
| **Hybrid** | `p_regime_0 > threshold` OR `pred_regime == 1` (Chop always allowed) |

Sweep thresholds 0.30 to 0.75 and compare Sharpe, Return, MaxDD vs the hard-label benchmarks.

In [ ]:
import contextlib, io as _io

def apply_prob_filter(df, regime_df, threshold, also_allow_chop=False):
    bull_conf = regime_df['p_regime_0'] > threshold
    if also_allow_chop:
        bull_conf = bull_conf | (regime_df['pred_regime'] == 1)
    mask = bull_conf.astype(int).reindex(df.index).fillna(1)
    d = df.copy()
    d['position'] = (d['position'] * mask).astype(int)
    return d

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

def run_sweep(also_allow_chop):
    rows = []
    for thr in thresholds:
        d = apply_prob_filter(df, regime_df, thr, also_allow_chop)
        with contextlib.redirect_stdout(_io.StringIO()):
            r = backtest(d, cost=COST, show_plot=False, save_html=None)
        overlap = df.index[df.index.isin(regime_df.index)]
        conf = regime_df['p_regime_0'].reindex(overlap) > thr
        if also_allow_chop:
            conf = conf | (regime_df['pred_regime'].reindex(overlap) == 1)
        blocked_pct = (~conf).mean() * 100
        cal_r = r['total_return'] / abs(r['max_drawdown']) if r['max_drawdown'] != 0 else 0
        rows.append({
            'threshold':   thr,
            'return':      r['total_return'] * 100,
            'sharpe':      r['sharpe_ratio'],
            'maxdd':       r['max_drawdown'] * 100,
            'calmar':      cal_r,
            'pf':          r['profit_factor'],
            'wr':          r['win_rate'] * 100,
            'trades':      r['num_trades'],
            'blocked_pct': blocked_pct,
            'df_variant':  d,
        })
    return rows

with contextlib.redirect_stdout(_io.StringIO()):
    r_base_m   = backtest(df_base,     cost=COST, show_plot=False, save_html=None)
    r_bullchop = backtest(df_bullchop,  cost=COST, show_plot=False, save_html=None)

rows_pure   = run_sweep(also_allow_chop=False)
rows_hybrid = run_sweep(also_allow_chop=True)

def _cal(r): return r['total_return'] / abs(r['max_drawdown']) if r['max_drawdown'] != 0 else 0

hdr = f'  {"Thr":>5}  {"Return":>9}  {"Sharpe":>7}  {"MaxDD":>8}  {"Calmar":>7}  {"PF":>6}  {"WR":>7}  {"Trades":>7}  {"Blocked%":>9}'
sep = '  ' + '-' * 88

def print_row(row):
    print(f'  {row["threshold"]:>5.2f}  {row["return"]:>8.1f}%  {row["sharpe"]:>7.2f}  '
          f'{row["maxdd"]:>7.1f}%  {row["calmar"]:>7.2f}  {row["pf"]:>6.2f}  '
          f'{row["wr"]:>6.1f}%  {row["trades"]:>7}  {row["blocked_pct"]:>8.1f}%')

print('=' * 96)
print('  PURE PROBABILITY THRESHOLD  (p_regime_0 > threshold)')
print('=' * 96)
print(hdr); print(sep)
for row in rows_pure:
    print_row(row)
print(f'  {"Base":>5}  {r_base_m["total_return"]*100:>8.1f}%  {r_base_m["sharpe_ratio"]:>7.2f}  '
      f'{r_base_m["max_drawdown"]*100:>7.1f}%  {_cal(r_base_m):>7.2f}  {r_base_m["profit_factor"]:>6.2f}  '
      f'{r_base_m["win_rate"]*100:>6.1f}%  {r_base_m["num_trades"]:>7}  {"0.0":>8}%')
print(f'  {"B+C":>5}  {r_bullchop["total_return"]*100:>8.1f}%  {r_bullchop["sharpe_ratio"]:>7.2f}  '
      f'{r_bullchop["max_drawdown"]*100:>7.1f}%  {_cal(r_bullchop):>7.2f}  {r_bullchop["profit_factor"]:>6.2f}  '
      f'{r_bullchop["win_rate"]*100:>6.1f}%  {r_bullchop["num_trades"]:>7}')

print()
print('=' * 96)
print('  HYBRID THRESHOLD  (p_regime_0 > threshold  OR  pred_regime == 1 Chop always allowed)')
print('=' * 96)
print(hdr); print(sep)
for row in rows_hybrid:
    print_row(row)


In [ ]:
best_pure   = max(rows_pure,   key=lambda r: r['sharpe'])
best_hybrid = max(rows_hybrid, key=lambda r: r['sharpe'])

print(f'Best pure threshold   : {best_pure["threshold"]:.2f}  '
      f'Sharpe {best_pure["sharpe"]:.2f}  Return {best_pure["return"]:.1f}%  MaxDD {best_pure["maxdd"]:.1f}%')
print(f'Best hybrid threshold : {best_hybrid["threshold"]:.2f}  '
      f'Sharpe {best_hybrid["sharpe"]:.2f}  Return {best_hybrid["return"]:.1f}%  MaxDD {best_hybrid["maxdd"]:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(thresholds, [r['sharpe'] for r in rows_pure],   'o-',  label='Pure p>thr',  color='#1565c0')
axes[0].plot(thresholds, [r['sharpe'] for r in rows_hybrid], 's--', label='Hybrid',       color='#e53935')
axes[0].axhline(r_base_m['sharpe_ratio'],   color='grey',    linestyle=':', label='Baseline')
axes[0].axhline(r_bullchop['sharpe_ratio'], color='#2e7d32', linestyle=':', label='Bull+Chop')
axes[0].set_xlabel('p_regime_0 threshold'); axes[0].set_ylabel('Sharpe ratio')
axes[0].set_title('Sharpe vs Threshold'); axes[0].legend(fontsize=8)

axes[1].plot(thresholds, [r['return'] for r in rows_pure],   'o-',  color='#1565c0')
axes[1].plot(thresholds, [r['return'] for r in rows_hybrid], 's--', color='#e53935')
axes[1].axhline(r_base_m['total_return']*100,   color='grey',    linestyle=':')
axes[1].axhline(r_bullchop['total_return']*100, color='#2e7d32', linestyle=':')
axes[1].set_xlabel('p_regime_0 threshold'); axes[1].set_ylabel('Total return %')
axes[1].set_title('Return vs Threshold')

axes[2].plot(thresholds, [r['maxdd'] for r in rows_pure],   'o-',  color='#1565c0')
axes[2].plot(thresholds, [r['maxdd'] for r in rows_hybrid], 's--', color='#e53935')
axes[2].axhline(r_base_m['max_drawdown']*100,   color='grey',    linestyle=':')
axes[2].axhline(r_bullchop['max_drawdown']*100, color='#2e7d32', linestyle=':')
axes[2].set_xlabel('p_regime_0 threshold'); axes[2].set_ylabel('Max drawdown %')
axes[2].set_title('MaxDD vs Threshold')

plt.suptitle('Probability threshold sweep — MoneyIn Long', fontsize=11)
plt.tight_layout()
plt.show()

# Equity curves: best threshold variants vs hard benchmarks
fig2, ax = plt.subplots(figsize=(15, 5))
bnh_eq = (1 + df['Close'].pct_change()).cumprod().dropna()

for label, d, col, ls in [
    ('Baseline',                           df_base,                   '#546e7a', '-'),
    ('Bull+Chop (hard)',                    df_bullchop,               '#2e7d32', '-'),
    (f'Pure thr={best_pure["threshold"]:.2f}',   best_pure['df_variant'],   '#1565c0', '-'),
    (f'Hybrid thr={best_hybrid["threshold"]:.2f}', best_hybrid['df_variant'], '#e53935', '--'),
]:
    eq = equity_curve(d, COST)
    ax.plot(eq.index, eq.values, label=label, color=col, linestyle=ls, linewidth=1.5)

ax.plot(bnh_eq.index, bnh_eq.values, label='BTC B&H', color='#bdbdbd', linestyle='--', alpha=0.6)
ax.axhline(1, color='black', linewidth=0.4, linestyle='--')
ax.set_ylabel('Cumulative return')
ax.set_title('Equity curves — best probability thresholds vs hard regime filters')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
